# LLM Evaluation: Metrics, Benchmarks & Frameworks

## Why Evaluation is Hard
- LLM outputs are open-ended no single correct answer
- Human evaluation is gold standard but slow/expensive
- Automated metrics correlate imperfectly with quality
- Models can overfit to benchmarks ("benchmark contamination")
- Different tasks need different metrics

## Automated Text Metrics

### BLEU (Bilingual Evaluation Understudy)
Measures n-gram precision between generated and reference text:

$$\text{BLEU} = BP \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

where $BP = \min(1, e^{1 - r/c})$ is the brevity penalty, $p_n$ is the modified n-gram precision.

### ROUGE (Recall-Oriented Understudy for Gisting Evaluation)
Measures recall of n-grams from reference in generated text:

$$\text{ROUGE-N} = \frac{\sum_{s \in \text{ref}} \sum_{\text{n-gram} \in s} \text{Count}_{match}(\text{n-gram})}{\sum_{s \in \text{ref}} \sum_{\text{n-gram} \in s} \text{Count}(\text{n-gram})}$$

### BERTScore
Uses BERT embeddings for semantic similarity:

$$P = \frac{1}{|\hat{y}|} \sum_{\hat{y}_j \in \hat{y}} \max_{y_i \in y} \mathbf{x}_{y_i}^T \mathbf{x}_{\hat{y}_j}$$

## Key Benchmarks

| Benchmark | What It Tests | Size |
|-----------|--------------|------|
| MMLU | 57-subject knowledge | 15K Q |
| HumanEval | Python code generation | 164 problems |
| GSM8K | Grade school math | 8.5K problems |
| MATH | Competition math | 12.5K problems |
| HellaSwag | Commonsense completion | 70K examples |
| ARC-Challenge | Science QA | 1.17K Q |
| TruthfulQA | Avoiding falsehoods | 817 Q |
| GPQA | PhD-level science | 448 Q |
| SWE-bench | Real GitHub issues | 2294 issues |

In [1]:
# BLEU, ROUGE, BERTScore implementation
# pip install rouge-score bert-score nltk
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
import nltk
nltk.download('punkt', quiet=True)

reference = "The cat sat on the mat near the window"
hypothesis = "A cat was sitting on a mat by the window"

# BLEU
ref_tokens = [reference.split()]
hyp_tokens = hypothesis.split()
smooth = SmoothingFunction().method1
bleu = sentence_bleu(ref_tokens, hyp_tokens, smoothing_function=smooth)
print(f'BLEU: {bleu:.4f}')

# ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores = scorer.score(reference, hypothesis)
for metric, score in scores.items():
    print(f'{metric}: P={score.precision:.3f}, R={score.recall:.3f}, F1={score.fmeasure:.3f}')

BLEU: 0.0561
rouge1: P=0.500, R=0.556, F1=0.526
rouge2: P=0.111, R=0.125, F1=0.118
rougeL: P=0.500, R=0.556, F1=0.526


In [2]:
# BERTScore
BERTSCORE_CODE = '''
from bert_score import score

references = ["The weather is nice today"]
hypotheses = ["It's a beautiful sunny day"]

P, R, F1 = score(hypotheses, references, lang='en', verbose=False)
print(f'BERTScore P: {P.mean():.4f}, R: {R.mean():.4f}, F1: {F1.mean():.4f}')
# Semantic similarity: 0.87 even though BLEU would be ~0.1
'''
print(BERTSCORE_CODE)


from bert_score import score

references = ["The weather is nice today"]
hypotheses = ["It's a beautiful sunny day"]

P, R, F1 = score(hypotheses, references, lang='en', verbose=False)
print(f'BERTScore P: {P.mean():.4f}, R: {R.mean():.4f}, F1: {F1.mean():.4f}')
# Semantic similarity: 0.87 even though BLEU would be ~0.1



In [3]:
# LLM-as-Judge using G-Eval pattern
from openai import OpenAI

def llm_judge(question, answer, criteria="coherence, accuracy, helpfulness"):
    client = OpenAI()
    prompt = f"""You are an expert evaluator. Score this answer on a scale of 1-10.

Question: {question}
Answer: {answer}

Evaluate on: {criteria}
Provide scores as JSON: {{"coherence": X, "accuracy": X, "helpfulness": X, "overall": X, "reasoning": "..."}}
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return response.choices[0].message.content

print('LLM-as-Judge function defined.')
# Usage:
# result = llm_judge("What is RAG?", "RAG combines retrieval with generation...")
# print(result)

LLM-as-Judge function defined.


In [4]:
# DeepEval modern LLM evaluation framework
DEEPEVAL_CODE = '''
# pip install deepeval
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, HallucinationMetric
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input="What is the capital of France?",
    actual_output="Paris is the capital of France.",
    expected_output="Paris",
    retrieval_context=["France is a country in Western Europe. Its capital is Paris."]
)

metrics = [
    AnswerRelevancyMetric(threshold=0.7),
    FaithfulnessMetric(threshold=0.7),
    HallucinationMetric(threshold=0.3),
]

evaluate([test_case], metrics)
'''
print(DEEPEVAL_CODE)


# pip install deepeval
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, HallucinationMetric
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input="What is the capital of France?",
    actual_output="Paris is the capital of France.",
    expected_output="Paris",
    retrieval_context=["France is a country in Western Europe. Its capital is Paris."]
)

metrics = [
    AnswerRelevancyMetric(threshold=0.7),
    FaithfulnessMetric(threshold=0.7),
    HallucinationMetric(threshold=0.3),
]

evaluate([test_case], metrics)



In [5]:
# Running LM Evaluation Harness (EleutherAI)
LM_EVAL_CODE = '''
# pip install lm_eval
# Run from command line:
# lm_eval --model hf --model_args pretrained=meta-llama/Llama-3.2-3B-Instruct \\
#         --tasks mmlu,hellaswag,arc_challenge \\
#         --device cuda:0 --batch_size 8

# Or programmatically:
import lm_eval

results = lm_eval.simple_evaluate(
    model="hf",
    model_args="pretrained=meta-llama/Llama-3.2-3B-Instruct",
    tasks=["mmlu", "hellaswag"],
    batch_size=8,
)
print(lm_eval.utils.make_table(results))
'''
print(LM_EVAL_CODE)


# pip install lm_eval
# Run from command line:
# lm_eval --model hf --model_args pretrained=meta-llama/Llama-3.2-3B-Instruct \
#         --tasks mmlu,hellaswag,arc_challenge \
#         --device cuda:0 --batch_size 8

# Or programmatically:
import lm_eval

results = lm_eval.simple_evaluate(
    model="hf",
    model_args="pretrained=meta-llama/Llama-3.2-3B-Instruct",
    tasks=["mmlu", "hellaswag"],
    batch_size=8,
)
print(lm_eval.utils.make_table(results))



## Additional Learning Resources

### Papers
- [MMLU benchmark](https://arxiv.org/abs/2009.03300)
- [HumanEval](https://arxiv.org/abs/2107.03374)
- [BLEU paper](https://aclanthology.org/P02-1040/)
- [BERTScore](https://arxiv.org/abs/1904.09675)
- [RAGAS](https://arxiv.org/abs/2309.15217)
- [G-Eval](https://arxiv.org/abs/2303.16634)

### Tools
- [EleutherAI lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)
- [RAGAS](https://github.com/explodinggradients/ragas)
- [DeepEval](https://github.com/confident-ai/deepeval)
- [promptfoo](https://github.com/promptfoo/promptfoo)
- [TruLens](https://github.com/truera/trulens)

### Leaderboards
- [Open LLM Leaderboard](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard)
- [LMSYS Chatbot Arena](https://chat.lmsys.org/)
- [BIG-bench](https://github.com/google/BIG-bench)